# Transformer 204M — Full Training Pipeline

**Auto-continue pipeline**: Pretrain → SFT → DPO

Works on:
- Google Colab (single T4 16GB)
- Kaggle (2× T4 16GB, DDP)

**Execution**: Run all cells top to bottom. The pipeline auto-detects environment and resumes from checkpoints.

## 1. Setup & Build

In [ ]:
%%bash
set -e

# ─── Auto-detect environment ─────────────────────────────────────────────
if [ -d /kaggle/working ]; then
    WORK_DIR="/kaggle/working/transformer_100m_cuda"
    echo "Environment: KAGGLE"
elif [ -d /content ]; then
    WORK_DIR="/content/transformer_100m_cuda"
    echo "Environment: COLAB"
else
    WORK_DIR="$(pwd)"
    echo "Environment: LOCAL"
fi
cd "$WORK_DIR"

export PATH="/usr/local/cuda/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda/lib64:$LD_LIBRARY_PATH"

# Detect GPU
GPU_NAME=$(nvidia-smi --query-gpu=name --format=csv,noheader | head -1)
echo "GPU: $GPU_NAME"
if   echo "$GPU_NAME" | grep -qi "T4";   then CUDA_ARCH=75
elif echo "$GPU_NAME" | grep -qi "V100"; then CUDA_ARCH=70
elif echo "$GPU_NAME" | grep -qi "A100"; then CUDA_ARCH=80
elif echo "$GPU_NAME" | grep -qi "L4";   then CUDA_ARCH=89
else CUDA_ARCH=75; fi

# Build
rm -rf build && mkdir -p build && cd build
USE_DDP_FLAG=""
if [ -d /kaggle/working ]; then USE_DDP_FLAG="-DUSE_DDP=ON"; fi
cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=$CUDA_ARCH $USE_DDP_FLAG
make -j$(nproc)
echo ""
ls -lh train_pretrain_cuda train_sft_cuda

# Kernel correctness check
echo ""
echo "Running grad_check..."
cmake --build . --target grad_check -j$(nproc)
./grad_check

# Build infer binary
echo ""
rm -rf "$WORK_DIR/build_infer"
mkdir -p "$WORK_DIR/build_infer" && cd "$WORK_DIR/build_infer"
cmake "$WORK_DIR/infer_cpp" -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=$CUDA_ARCH
make -j$(nproc)
echo "=== ALL BUILDS OK ==="

## 2. Configuration

Edit the variables below to match your setup.

In [ ]:
import os

# ─── Auto-detect environment ─────────────────────────────────────────────
if os.path.exists("/kaggle/working"):
    ENV = "kaggle"
    WORK_DIR = "/kaggle/working/transformer_100m_cuda"
    CKPT_DIR = "/kaggle/working/checkpoints"
    DATA_DIR = "/kaggle/input/datasets/naufalrachmat/datatraining/data/corpus_final"
elif os.path.exists("/content"):
    ENV = "colab"
    WORK_DIR = "/content/transformer_100m_cuda"
    CKPT_DIR = f"{WORK_DIR}/checkpoints"
    DATA_DIR = "data/corpus_final"
else:
    ENV = "local"
    WORK_DIR = os.getcwd()
    CKPT_DIR = f"{WORK_DIR}/checkpoints"
    DATA_DIR = "data/corpus_final"

os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Environment: {ENV}")
print(f"Work dir:    {WORK_DIR}")
print(f"Checkpoints: {CKPT_DIR}")

## 3. Run Full Pipeline (Pretrain → SFT → DPO)

This cell runs the full auto-continue pipeline. It:
1. Starts/resumes pretraining until `max_tokens` reached
2. Automatically starts SFT using the pretrain checkpoint
3. Automatically starts DPO using the SFT checkpoint

Each stage is independently resumable. Kill and re-run to resume from last checkpoint.

In [ ]:
import subprocess, os

# ─── Pipeline configuration ──────────────────────────────────────────────
STAGES = "pretrain,sft,dpo"  # Which stages to run (comma-separated)

# Pretrain config
PRETRAIN_BATCH = 35 if ENV == "kaggle" else 24
PRETRAIN_LR = "1.4e-4"
MAX_TOKENS = "2000000000"  # 2B

# SFT config
SFT_LR = "3e-5"
SFT_EPOCHS = "1"
SFT_BATCH = "8"
SFT_MAX_SEQ = "512"

# DPO config
DPO_LR = "1e-5"
DPO_EPOCHS = "3"
DPO_BETA = "0.1"
DPO_BATCH = "4"

# ─── Build command ────────────────────────────────────────────────────────
cmd = [
    'python3', 'scripts/train_pipeline.py',
    '--auto_continue',
    '--stages',          STAGES,
    '--checkpoint_dir',  CKPT_DIR,
    '--data_dir',        DATA_DIR,
    '--vocab_dir',       'vocab_6',
    '--bpe',             'bpe_v6.bin',
    # Pretrain
    '--batch_size',      str(PRETRAIN_BATCH),
    '--max_seq',         '256',
    '--max_tokens',      MAX_TOKENS,
    '--lr',              PRETRAIN_LR,
    '--grad_clip',       '5.0',
    '--loss_scale',      '4096',
    '--safety',          'light',
    '--save_every',      '150',
    '--upload_every',    '300',
    '--eval_every_tokens', '200000000',
    '--eval_infer_bin',  './build_infer/infer',
    # SFT
    '--sft_lr',          SFT_LR,
    '--sft_epochs',      SFT_EPOCHS,
    '--sft_batch_size',  SFT_BATCH,
    '--sft_max_seq',     SFT_MAX_SEQ,
    '--sft_dir',         'sft_data/sft_translation_focused',
    # DPO
    '--dpo_lr',          DPO_LR,
    '--dpo_epochs',      DPO_EPOCHS,
    '--dpo_beta',        DPO_BETA,
    '--dpo_batch_size',  DPO_BATCH,
    '--dpo_dir',         'sft_data/dpo',
    '--resume',
]

# Kaggle DDP
if ENV == "kaggle":
    cmd += ['--ddp', '--num_gpus', '2']

# ─── Run pipeline ─────────────────────────────────────────────────────────
env = os.environ.copy()
env.pop('CUDA_VISIBLE_DEVICES', None)
if ENV == "kaggle":
    env['NCCL_P2P_DISABLE'] = '1'

print(f"Running: {' '.join(cmd[:8])}...")
print()

proc = subprocess.Popen(
    cmd,
    cwd=WORK_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=0,
)

try:
    for raw in iter(proc.stdout.readline, b''):
        print(raw.decode('utf-8', errors='replace'), end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print("\nPipeline interrupted. Re-run to resume.")

proc.wait()
print(f"\nPipeline exit code: {proc.returncode}")

## 4. (Alternative) Run Single Stage

Use this if you want to run just SFT or DPO independently.

In [ ]:
# ─── Run SFT only ─────────────────────────────────────────────────────────
# Change STAGE to "dpo" to run DPO only.
STAGE = "sft"  # or "dpo"

cmd = [
    'python3', 'scripts/train_pipeline.py',
    '--stages', STAGE,
    '--checkpoint_dir', CKPT_DIR,
    '--vocab_dir', 'vocab_6',
    '--bpe', 'bpe_v6.bin',
    '--sft_lr', '3e-5',
    '--sft_epochs', '1',
    '--sft_batch_size', '8',
    '--sft_max_seq', '512',
    '--sft_dir', 'sft_data/sft_translation_focused',
    '--dpo_lr', '1e-5',
    '--dpo_epochs', '3',
    '--dpo_beta', '0.1',
    '--dpo_batch_size', '4',
    '--dpo_dir', 'sft_data/dpo',
    '--resume',
]
if ENV == "kaggle":
    cmd += ['--ddp', '--num_gpus', '2']

env = os.environ.copy()
env.pop('CUDA_VISIBLE_DEVICES', None)

proc = subprocess.Popen(cmd, cwd=WORK_DIR, env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
try:
    for raw in iter(proc.stdout.readline, b''):
        print(raw.decode('utf-8', errors='replace'), end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
proc.wait()
print(f"\nExit code: {proc.returncode}")